In [1]:
!pip install protobuf==3.20.3

     ---------------------------------------- 0.0/904.0 kB ? eta -:--:--
     -- ------------------------------------ 61.4/904.0 kB 1.7 MB/s eta 0:00:01
     -------- ----------------------------- 204.8/904.0 kB 2.1 MB/s eta 0:00:01
     ---------------- --------------------- 399.4/904.0 kB 2.8 MB/s eta 0:00:01
     ---------------------------------- --- 809.0/904.0 kB 4.6 MB/s eta 0:00:01
     -------------------------------------- 904.0/904.0 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.1
    Uninstalling protobuf-6.33.1:
      Successfully uninstalled protobuf-6.33.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 3.20.3 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import gc

In [ ]:
IMG_SIZE = 150
BATCH_SIZE = 32
EPOCHS = 20
KAGGLE_PATH = '/kaggle/input/processed-img-2/processed/train'
# Đường dẫn local
LOCAL_PATH = os.path.join('..', 'data', 'processed', 'train')

if os.path.exists(KAGGLE_PATH):
    print("MÔI TRƯỜNG: KAGGLE GPU")
    dataset_path = KAGGLE_PATH
elif os.path.exists(LOCAL_PATH):
    print("MÔI TRƯỜNG: LOCAL PC")
    dataset_path = LOCAL_PATH
else:
    raise FileNotFoundError("❌ Không tìm thấy dữ liệu! Hãy Add Data trên Kaggle.")

print(f"Đang đọc dữ liệu từ: {dataset_path}")
data = []
labels = []
categories = ['NORMAL', 'PNEUMONIA']

for category in categories:
    path = os.path.join(dataset_path, category)
    class_num = categories.index(category)
    try:
        files = os.listdir(path)
        if 'kaggle' not in dataset_path.lower():
            files = files[:100]
            
        for img_name in files:
            try:
                img_path = os.path.join(path, img_name)
                img_array = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img_array is not None:
                    resized_arr = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                    data.append(resized_arr)
                    labels.append(class_num)
            except Exception as e:
                pass
    except Exception as e:
        print(f"Lỗi đọc folder {category}: {e}")

print(f"Đã load xong: {len(data)} ảnh.")

In [ ]:
X = np.array(data)
y = np.array(labels)

# Giải phóng RAM
del data
del labels
gc.collect()

# Chuẩn hóa 0-255 -> 0-1
X = X / 255.0
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

# Chia tập Train/Val (80/20)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Train set: {X_train.shape}")
print(f"📊 Val set:   {X_val.shape}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Dự đoán trên tập Test...")

predictions = model_best.predict(X_test_new)
y_pred = (predictions >= 0.5).astype(int)

acc = accuracy_score(y_test_new, y_pred)

print("\n" + "="*40)
print(f"Kết quả: {acc*100:.2f}%")
print("="*40)

print("Classification Report:")
print(classification_report(y_test_new, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

cm = confusion_matrix(y_test_new, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['NORMAL', 'PNEUMONIA'], 
            yticklabels=['NORMAL', 'PNEUMONIA'])
plt.xlabel('Predicted')
plt.ylabel('Thực tế (Actual)')
plt.title('Confusion Matrix (Ma trận nhầm lẫn)')
plt.show()